# Live Social Media Dataset From Reddit

Source: https://www.reddit.com

Summary: Fetch real-time data from Reddit and generate daily sentiment scores using our pretrained sentiment model.

For setup:
1. Upload all json files before running the code.
  - algotrading
  - Cryptocurrency
  - CryptoMarkets
  - dividends
  - economy
  - ETFs
  - financialindependence
  - FinancialPlanning
  - Forex
  - investing
  - options
  - pennystocks
  - personalfinance
  - quant
  - RobinHood
  - SecurityAnalysis
  - SPACs
  - StockMarket
  - stocks
  - Superstonk
  - ValueInvesting
  - wallstreetbets

2. Upload sentiment_model.zip before running the code.
3. By default, each run retrieves only the latest 100 rows of data. To capture a broader range of S&P 500 related content, specify multiple relevant subreddits so more data can be collected.

# Step 1: Fetch real-time data from Reddit

In [3]:
import json
import pandas as pd
from datetime import datetime
import os

def process_local_reddit_file(filename):
    if not os.path.exists(filename):
        print(f"Warning: {filename} not found. Skipping.")
        return None
    with open(filename, 'r') as f:
        raw_data = json.load(f)

    posts = raw_data['data']['children']
    data = []

    for post in posts:
        p = post['data']
        # Extracting columns to match emilpartow/reddit_finance_posts_sp500 schema
        data.append({
            "id": p.get('id'),
            "title": p.get('title'),
            "text": p.get('selftext'),
            "created_utc": p.get('created_utc'),
            "author": p.get('author'),
            "score": p.get('score'),
            "num_comments": p.get('num_comments'),
            "upvote_ratio": p.get('upvote_ratio'),
            "flair": p.get('link_flair_text'),
            "full_link": f"https://www.reddit.com{p.get('permalink')}",
            "subreddit": p.get('subreddit'),
        })

    df = pd.DataFrame(data)

    # Apply Group 13 specific logic
    # Calendar Alignment format (DD/MM/YYYY)
    df['created_datetime'] = pd.to_datetime(df['created_utc'], unit='s').dt.strftime('%d/%m/%Y')

    return df

subreddits = [
    "algotrading", "Cryptocurrency", "CryptoMarkets", "dividends", "economy",
    "ETFs", "financialindependence", "FinancialPlanning", "Forex", "investing",
    "options", "pennystocks", "personalfinance", "quant", "RobinHood",
    "SecurityAnalysis", "SPACs", "StockMarket", "stocks", "Superstonk",
    "ValueInvesting", "wallstreetbets"
]

# Create a list of DataFrames by processing each .json file
# This assumes the files are named 'stocks.json', 'wallstreetbets.json', etc.
all_dfs = []
for sub in subreddits:
    file_path = f"{sub}.json"
    df_sub = process_local_reddit_file(file_path)
    if df_sub is not None:
        all_dfs.append(df_sub)

# Concatenate all individual DataFrames into one
if all_dfs:
    df_final = pd.concat(all_dfs, ignore_index=True)
    print(f"Successfully combined {len(all_dfs)} subreddits.")
    print(f"Total rows: {len(df_final)}")

    # Optional: Save to a single CSV
    df_final.to_csv("combined_reddit_finance.csv", index=False)
else:
    print("No data found to combine.")

Successfully combined 22 subreddits.
Total rows: 2200


Step 2: Predict daily sentiment for dataset and Export results

In [4]:
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from tqdm import tqdm
import zipfile
import os
import matplotlib.pyplot as plt

def run_full_sentiment_pipeline(input_csv='combined_reddit_finance.csv', zip_path='sentiment_model.zip'):
    model_dir = './sentiment_model'

    # Extraction Check
    if not os.path.exists(model_dir):
        if os.path.exists(zip_path):
            try:
                print(f"Unzipping {zip_path}...")
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall('.')
                print("Extraction complete.")
            except Exception as e:
                print(f"Zip Error: {e}")
                return
        else:
            print(f"Missing: {model_dir} or {zip_path}")
            return

    # Load Data
    print(f"Loading {input_csv}...")
    df = pd.read_csv(input_csv)
    df['combined_text'] = df['title'].fillna('') + " " + df['text'].fillna('')

    # Handle dates from UTC or string
    if 'created_utc' in df.columns:
        df['date'] = pd.to_datetime(df['created_utc'], unit='s').dt.normalize()
    else:
        df['date'] = pd.to_datetime(df['created_datetime'], dayfirst=True).dt.normalize()

    # Model Inference
    print("Loading Model weights...")
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()

    results = []
    print(f"Predicting Sentiment (Detected {model.config.num_labels} classes)...")

    with torch.no_grad():
        for text in tqdm(df['combined_text']):
            inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
            outputs = model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1).cpu().numpy()[0]

            # Auto-adjust based on whether it is trained for 2 or 3 classes
            if model.config.num_labels == 3:
                score = (probs[2] * 1.0) + (probs[1] * 0.5)
            else:
                score = probs[1] # Probability of Positive class

            results.append({
                'sentiment_score': score,
                'label': torch.argmax(outputs.logits, dim=-1).item()
            })

    df = pd.concat([df, pd.DataFrame(results)], axis=1)

    # Generate all 7 Files
    print("Generating project files...")

    # 1. predictions.csv
    df.to_csv("predictions.csv", index=False)

    # 2. daily.csv
    daily = df.groupby('date').agg({'sentiment_score': 'mean', 'combined_text': 'count'}).rename(columns={'combined_text': 'post_count'})
    daily.to_csv("daily.csv")

    # 3. daily_sentiment_results.csv
    daily_res = daily.copy()
    daily_res['SMO'] = daily_res['sentiment_score'].rolling(window=5).mean() - \
                       daily_res['sentiment_score'].rolling(window=20).mean()
    daily_res.to_csv("daily_sentiment_results.csv")

    # 4. sentiment_trend.png
    plt.figure(figsize=(12,6))
    plt.plot(daily_res.index, daily_res['sentiment_score'], label='Daily Sentiment', alpha=0.4)
    plt.plot(daily_res.index, daily_res['sentiment_score'].rolling(window=7).mean(), label='7-Day Trend', color='red')
    plt.title("S&P 500 Sentiment Analysis - April 2026")
    plt.legend()
    plt.savefig("sentiment_trend.png")
    plt.close()

    # 5. test_predictions_full.csv
    df[['date', 'sentiment_score', 'label']].to_csv("test_predictions_full.csv", index=False)

    # 6. model_performance_summary.txt
    with open("model_performance_summary.txt", "w") as f:
        f.write(f"Processed: {len(df)} posts\nAvg Sentiment: {df['sentiment_score'].mean():.4f}\n")

    # 7. sentiment_model/ exists as the directory from step 1
    print("\n All 7 files successfully generated.")

# Run the pipeline
if __name__ == "__main__":
    run_full_sentiment_pipeline()

Loading combined_reddit_finance.csv...
Loading Model weights...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Predicting Sentiment (Detected 2 classes)...


100%|██████████| 2200/2200 [14:45<00:00,  2.48it/s]


Generating project files...

 All 7 files successfully generated.
